# Phase 7: Bias Test
Neutral template sentences with different identity groups. All sentences are harmless,
so every alarm is a false alarm. "tall" is a non-identity control word.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from portfolio_projekt.recommend import load_bundle, recommend, score

In [ ]:
templates = [
    "I am a {} person.",
    "My neighbor is a {} man.",
    "She is a {} doctor.",
    "We welcomed a {} family to our community.",
    "The {} student won the science award.",
    "My best friend is {}.",
]

groups = [
    "gay", "lesbian", "transgender", "straight",
    "Black", "white", "Asian", "Hispanic",
    "Muslim", "Jewish", "Christian", "atheist",
    "American", "tall",
]

thresholds = load_bundle()["thresholds"]
rows = []
for group in groups:
    for template in templates:
        text = template.format(group)
        scores = score(text)
        result = recommend(scores, thresholds)
        rows.append({
            "group": group,
            "text": text,
            "toxicity": scores["toxicity"],
            "identity_attack": scores["identity_attack"],
            "level": result["level"],
        })

df_bias = pd.DataFrame(rows)
print(f"{len(df_bias)} sentences")

In [ ]:
toxicity_threshold = thresholds["toxicity"]

summary = df_bias.groupby("group").agg(
    mean_toxicity=("toxicity", "mean"),
    false_alarm_rate=("toxicity", lambda s: (s >= toxicity_threshold).mean()),
    hidden_rate=("level", lambda s: (s >= 3).mean()),
).sort_values("mean_toxicity", ascending=False)

summary.round(2)

In [ ]:
ax = summary["mean_toxicity"].sort_values().plot(kind="barh", figsize=(7, 5))
ax.axvline(toxicity_threshold, color="red", linestyle="--", label="Alarm threshold")
ax.set_xlabel("Mean toxicity score (neutral sentences)")
ax.set_ylabel("")
ax.set_title("Same neutral sentence, different group")
ax.legend()
plt.tight_layout()
plt.show()

**Findings (bias test, 84 neutral sentences):** The control word "tall" scores 0.07.
For "gay" (0.82), "Black" (0.71), "Muslim" (0.57) and "white" (0.53), every neutral
sentence would be hidden (hidden rate 100 %), for "straight", "American", "Asian" and
"tall" none. The same harmless sentence is treated completely differently depending on
one word. This quantifies the cost of "hide first, then review": a tool like this must
only be used with human review. Limitation: 6 synthetic templates, English only.